In [1]:
import sys
print(sys.executable)

C:\Dev\data-exercices\phase-1\s1-s2-python-donnees-financieres\.venv\Scripts\python.exe


# Analyse de cours boursiers fictifs

**Exercice S1–S2 — Phase 1**
Données fictives : Apple, Tesla, Amazon

## Sommaire
1. Création des données fictives
2. Calcul de la variation journalière
3. Moyenne mobile sur 7 jours
4. Min / max
5. Fonctions réutilisables

In [3]:
import random
from datetime import date, timedelta

random.seed(42)

def generer_cours_fictifs(action: str, prix_initial: float, nb_jours: int = 30) -> list[dict]:
    """
    Génère une série de cours boursiers fictifs pour une action donnée.

    Args:
        action: nom de l'action (ex: "Apple")
        prix_initial: prix de départ en dollars
        nb_jours: nombre de jours à générer (30 par défaut)

    Returns:
        Liste de dictionnaires, un par jour, avec date/action/prix
    """
    cours = []
    prix_actuel = prix_initial
    date_actuelle = date(2026, 1, 1)

    for _ in range(nb_jours):
        variation = random.uniform(-3, 3)
        prix_actuel = round(prix_actuel + variation, 2)

        cours.append({
            "date": date_actuelle.isoformat(),
            "action": action,
            "prix": prix_actuel
        })

        date_actuelle += timedelta(days=1)

    return cours

In [4]:
cours_apple = generer_cours_fictifs("Apple", 150.00)
cours_tesla = generer_cours_fictifs("Tesla", 220.00)
cours_amazon = generer_cours_fictifs("Amazon", 145.00)

tous_les_cours = cours_apple + cours_tesla + cours_amazon

print(f"Nombre total d'enregistrements : {len(tous_les_cours)}")
print(tous_les_cours[:3])

Nombre total d'enregistrements : 90
[{'date': '2026-01-01', 'action': 'Apple', 'prix': 148.47}, {'date': '2026-01-02', 'action': 'Apple', 'prix': 146.31}, {'date': '2026-01-03', 'action': 'Apple', 'prix': 143.92}]


In [8]:
def calculer_variations(cours: list[dict]) -> list[dict]:
    """
    Ajoute la variation journalière (absolue en $ et en %) à chaque enregistrement.
    Le premier jour de chaque action n'a pas de variation (pas de jour précédent).
    """
    cours_avec_variation = []
    prix_precedent = None
    action_precedente = None

    for enregistrement in cours:
        enregistrement = enregistrement.copy()
        action_actuelle = enregistrement["action"]

        if action_actuelle == action_precedente:
            variation = round(enregistrement["prix"] - prix_precedent, 2)
            variation_pct = round((variation / prix_precedent) * 100, 2)
        else:
            variation = None
            variation_pct = None

        enregistrement["variation"] = variation
        enregistrement["variation_pct"] = variation_pct
        cours_avec_variation.append(enregistrement)

        prix_precedent = enregistrement["prix"]
        action_precedente = action_actuelle

    return cours_avec_variation

tous_les_cours = calculer_variations(tous_les_cours)

In [9]:
print(tous_les_cours[0])   # premier jour Apple : variation doit être None
print(tous_les_cours[1])   # deuxième jour Apple : variation doit exister
print(tous_les_cours[29])  # dernier jour Apple
print(tous_les_cours[30])  # premier jour Tesla : variation doit être None aussi

{'date': '2026-01-01', 'action': 'Apple', 'prix': 148.47, 'variation': None, 'variation_pct': None}
{'date': '2026-01-02', 'action': 'Apple', 'prix': 146.31, 'variation': -2.16, 'variation_pct': -1.45}
{'date': '2026-01-30', 'action': 'Apple', 'prix': 131.22, 'variation': -0.24, 'variation_pct': -0.18}
{'date': '2026-01-01', 'action': 'Tesla', 'prix': 217.75, 'variation': None, 'variation_pct': None}


In [10]:
def calculer_moyenne_mobile(cours: list[dict], fenetre: int = 7) -> list[dict]:
    """
    Ajoute une moyenne mobile glissante sur `fenetre` jours à chaque enregistrement.
    None tant que l'action n'a pas encore `fenetre` jours d'historique.
    """
    historique_par_action = {}
    resultat = []

    for enregistrement in cours:
        enregistrement = enregistrement.copy()
        action = enregistrement["action"]

        historique = historique_par_action.setdefault(action, [])
        historique.append(enregistrement["prix"])

        if len(historique) > fenetre:
            historique.pop(0)

        if len(historique) == fenetre:
            moyenne = round(sum(historique) / fenetre, 2)
        else:
            moyenne = None

        enregistrement["moyenne_mobile_7j"] = moyenne
        resultat.append(enregistrement)

    return resultat

In [11]:
tous_les_cours = calculer_moyenne_mobile(tous_les_cours)

print(tous_les_cours[0])   # jour 1 Apple : None (1 seul jour d'historique)
print(tous_les_cours[5])   # jour 6 Apple : None (6 jours d'historique, pas encore 7)
print(tous_les_cours[6])   # jour 7 Apple : première vraie moyenne mobile
print(tous_les_cours[30])  # jour 1 Tesla : None (nouvelle action, historique reparti à zéro)

{'date': '2026-01-01', 'action': 'Apple', 'prix': 148.47, 'variation': None, 'variation_pct': None, 'moyenne_mobile_7j': None}
{'date': '2026-01-06', 'action': 'Apple', 'prix': 146.17, 'variation': 0.54, 'variation_pct': 0.37, 'moyenne_mobile_7j': None}
{'date': '2026-01-07', 'action': 'Apple', 'prix': 143.36, 'variation': -2.81, 'variation_pct': -1.92, 'moyenne_mobile_7j': 145.6}
{'date': '2026-01-01', 'action': 'Tesla', 'prix': 217.75, 'variation': None, 'variation_pct': None, 'moyenne_mobile_7j': None}


In [12]:
def calculer_min_max(cours: list[dict]) -> dict:
    """
    Calcule le prix minimum et maximum observés pour chaque action,
    avec la date à laquelle chaque extremum a été atteint.
    """
    stats_par_action = {}

    for enregistrement in cours:
        action = enregistrement["action"]
        prix = enregistrement["prix"]
        jour = enregistrement["date"]

        if action not in stats_par_action:
            stats_par_action[action] = {
                "min": prix, "date_min": jour,
                "max": prix, "date_max": jour,
            }
        else:
            stats = stats_par_action[action]
            if prix < stats["min"]:
                stats["min"] = prix
                stats["date_min"] = jour
            if prix > stats["max"]:
                stats["max"] = prix
                stats["date_max"] = jour

    return stats_par_action

In [13]:
stats = calculer_min_max(tous_les_cours)

for action, valeurs in stats.items():
    print(f"{action} — min: {valeurs['min']} ({valeurs['date_min']}) | max: {valeurs['max']} ({valeurs['date_max']})")

Apple — min: 131.22 (2026-01-30) | max: 148.47 (2026-01-01)
Tesla — min: 214.33 (2026-01-09) | max: 225.62 (2026-01-22)
Amazon — min: 141.22 (2026-01-04) | max: 150.02 (2026-01-28)


## Conclusions

- **Apple** a été l'action la plus volatile à la baisse sur la période, passant de 150$ à un plus bas de 131.22$ (30 janvier).
- **Tesla** a montré la plus grande amplitude en valeur absolue (214.33$ à 225.62$, soit ~11$ d'écart).
- **Amazon** est restée la plus stable des trois, avec une fourchette resserrée entre 141.22$ et 150.02$.

Ces résultats sont générés à partir de données **fictives** (graine aléatoire fixe = 42, reproductible), à but pédagogique uniquement.